# Pivot tables for heatmaps

## Load data

In [2]:
path = '../../data/EIA/fuel_type_data_california.parquet'

In [3]:
import pandas as pd
df = pd.read_parquet(path).set_index('period').sort_index()

df = df.loc['2024', ['type-name', 'fueltype', 'value']]
df.columns = ['technology', 'tech', 'energy']

df

,technology,tech,energy
period,,,
2024-01-01 00:00:00-07:00,Wind,WND,243
2024-01-01 00:00:00-07:00,Hydro,WAT,3541
...,...,...,...
2024-12-31 23:00:00-07:00,Hydro,WAT,4125
2024-12-31 23:00:00-07:00,Wind,WND,476


## Calculate temporal properties

In [4]:
from modules import utils
df = utils.add_time_features(df)

df

ModuleNotFoundError: No module named 'modules'

In [5]:
df["year"] = df.index.year
df["month"]= df.index.month
df["day"]= df.index.day
df["hour"]= df.index.hour
df["weekday"]= df.index.weekday
df["weekend"]= df.index.dayofweek > 4

## Steps
### Aggregate data with pivot table

In [6]:
df.pivot_table?

Signature:
df.pivot_table(
    values=None,
    index=None,
    columns=None,
    aggfunc: 'AggFuncType' = 'mean',
    fill_value=None,
    margins: 'bool' = False,
    dropna: 'bool' = True,
    margins_name: 'Level' = 'All',
    observed: 'bool | lib.NoDefault' = <no_default>,
    sort: 'bool' = True,
) -> 'DataFrame'
Docstring:
Create a spreadsheet-style pivot table as a DataFrame.

The levels in the pivot table will be stored in MultiIndex objects
(hierarchical indexes) on the index and columns of the result DataFrame.

Parameters
----------
values : list-like or scalar, optional
    Column or columns to aggregate.
index : column, Grouper, array, or list of the previous
    Keys to group by on the pivot table index. If a list is passed,
    it can contain any of the other types (except list). If an array is
    passed, it must be the same length as the data and will be used in
    the same manner as column values.
columns : column, Grouper, array, or list of the previous
    Keys t

In [8]:
df[["tech", "technology"]].drop_duplicates().style

,tech,technology
period,,
2024-01-01 00:00:00-07:00,WND,Wind
2024-01-01 00:00:00-07:00,WAT,Hydro
2024-01-01 00:00:00-07:00,SUN,Solar
2024-01-01 00:00:00-07:00,OTH,Other
2024-01-01 00:00:00-07:00,OIL,Petroleum
2024-01-01 00:00:00-07:00,NUC,Nuclear
2024-01-01 00:00:00-07:00,NG,Natural Gas
2024-01-01 00:00:00-07:00,COL,Coal


In [15]:
c_num = "energy"
c_cat_idx = "hour"
c_cat_col = "tech"

r = df.pivot_table(
    values=c_num,
    index=c_cat_idx,
    columns=c_cat_col,
    aggfunc='mean',
)

### Style background gradient to visualize heatmatrix

In [12]:
r.style.background_gradient?

Signature:
r.style.background_gradient(
    cmap: 'str | Colormap' = 'PuBu',
    low: 'float' = 0,
    high: 'float' = 0,
    axis: 'Axis | None' = 0,
    subset: 'Subset | None' = None,
    text_color_threshold: 'float' = 0.408,
    vmin: 'float | None' = None,
    vmax: 'float | None' = None,
    gmap: 'Sequence | None' = None,
) -> 'Styler'
Docstring:
Color the background in a gradient style.

The background color is determined according
to the data in each column, row or frame, or by a given
gradient map. Requires matplotlib.

Parameters
----------
cmap : str or colormap
    Matplotlib colormap.
low : float
    Compress the color range at the low end. This is a multiple of the data
    range to extend below the minimum; good values usually in [0, 1],
    defaults to 0.
high : float
    Compress the color range at the high end. This is a multiple of the data
    range to extend above the maximum; good values usually in [0, 1],
    defaults to 0.
axis : {0, 1, "index", "columns", Non

In [16]:
r.style.background_gradient(cmap="Greens")

tech,COL,NG,NUC,OIL,OTH,SUN,WAT,WND
hour,,,,,,,,
0,431.778689,12735.718579,2098.482192,59.495890,2284.161202,11.571038,4628.759563,2876.368852
1,413.622951,12283.459016,2098.980822,59.490411,1058.571038,8.213115,4251.715847,2836.516393
2,395.224044,11735.314208,2097.635616,59.424658,682.833333,7.740437,3883.505464,2787.431694
3,371.005464,11218.885246,2095.868493,59.438356,452.685792,6.030055,3593.639344,2728.508197
4,357.024590,10901.158470,2094.158904,59.282192,231.581967,4.516393,3436.330601,2660.355191
5,354.551913,10753.027322,2092.895890,59.424658,139.472678,1.833333,3398.590164,2583.808743
6,367.825137,10680.161202,2092.816438,59.200000,298.065574,-0.524590,3504.866120,2477.743169
7,402.002732,10795.232240,2092.586301,59.079452,816.702186,72.177596,3549.292350,2366.590164
8,407.319672,11039.775956,2092.498630,58.605479,1290.199454,1164.497268,3503.945355,2223.909836


## Customize heatmatrix

### Format numbers

In [17]:
(r
 .div(1000)
 .style
    .format(precision=2)
    .background_gradient(cmap="Greens")
)

tech,COL,NG,NUC,OIL,OTH,SUN,WAT,WND
hour,,,,,,,,
0,0.43,12.74,2.10,0.06,2.28,0.01,4.63,2.88
1,0.41,12.28,2.10,0.06,1.06,0.01,4.25,2.84
2,0.40,11.74,2.10,0.06,0.68,0.01,3.88,2.79
3,0.37,11.22,2.10,0.06,0.45,0.01,3.59,2.73
4,0.36,10.90,2.09,0.06,0.23,0.00,3.44,2.66
5,0.35,10.75,2.09,0.06,0.14,0.00,3.40,2.58
6,0.37,10.68,2.09,0.06,0.30,-0.00,3.50,2.48
7,0.40,10.80,2.09,0.06,0.82,0.07,3.55,2.37
8,0.41,11.04,2.09,0.06,1.29,1.16,3.50,2.22


### Axis

In [29]:
(r
 .div(1000)
 .style
    .format(precision=2)
    .background_gradient(cmap="Greens", axis=None) # axis = 0 for columns, axis = 1 for rows.
)

tech,COL,NG,NUC,OIL,OTH,SUN,WAT,WND
hour,,,,,,,,
0,0.43,12.74,2.10,0.06,2.28,0.01,4.63,2.88
1,0.41,12.28,2.10,0.06,1.06,0.01,4.25,2.84
2,0.40,11.74,2.10,0.06,0.68,0.01,3.88,2.79
3,0.37,11.22,2.10,0.06,0.45,0.01,3.59,2.73
4,0.36,10.90,2.09,0.06,0.23,0.00,3.44,2.66
5,0.35,10.75,2.09,0.06,0.14,0.00,3.40,2.58
6,0.37,10.68,2.09,0.06,0.30,-0.00,3.50,2.48
7,0.40,10.80,2.09,0.06,0.82,0.07,3.55,2.37
8,0.41,11.04,2.09,0.06,1.29,1.16,3.50,2.22


## Rankings

### Calculate margins

In [50]:
c_num = "energy"
c_cat_idx = "technology"
c_cat_col = "month"

margins_name = "AVG"

r = df.pivot_table(
    values=c_num,
    index=c_cat_idx,
    columns=c_cat_col,
    aggfunc='mean',
    margins = True,
    margins_name = margins_name
)

r

month,1,2,3,4,5,6,7,8,9,10,11,12,AVG
technology,,,,,,,,,,,,,
Coal,256.860215,231.339080,218.561828,137.995833,196.044355,160.919444,242.415323,405.372312,675.858333,782.419355,692.220833,765.732527,397.836749
Hydro,2390.387097,3555.600575,4087.971774,4153.802778,4769.854839,4114.776389,4353.079301,4275.838710,3497.786111,2255.840054,1872.802778,2004.024194,3444.082878
...,...,...,...,...,...,...,...,...,...,...,...,...,...
Wind,1555.297043,1890.843391,2666.627688,2700.608333,3581.381720,3376.955556,2436.137097,2623.821237,2112.944444,1827.712366,1843.634722,1667.287634,2357.827755
AVG,3138.347782,2922.728269,2801.117608,2792.628646,3085.341734,3501.494444,4225.868616,4058.272345,3707.622049,3351.705477,2798.009104,2834.939516,3271.129543


### Sort rows to rank table

In [51]:
(r
 .sort_values(by = 12, ascending = False)
 .style
    .format(precision=2)
)

month,1,2,3,4,5,6,7,8,9,10,11,12,AVG
technology,,,,,,,,,,,,,
Natural Gas,14972.37,11002.23,8051.68,7367.16,6950.15,9814.25,16667.44,15026.62,13989.91,13773.56,11058.15,12027.18,11741.77
Solar,3094.03,3858.12,4637.88,6140.75,7324.81,7880.85,7355.27,7431.68,6854.19,5566.70,4100.21,3587.51,5656.01
AVG,3138.35,2922.73,2801.12,2792.63,3085.34,3501.49,4225.87,4058.27,3707.62,3351.71,2798.01,2834.94,3271.13
Nuclear,2261.29,2259.54,2265.05,1360.05,1401.93,2271.05,2265.42,2225.15,2108.20,2194.74,2266.16,2262.51,2094.75
Hydro,2390.39,3555.60,4087.97,4153.80,4769.85,4114.78,4353.08,4275.84,3497.79,2255.84,1872.80,2004.02,3444.08
Wind,1555.30,1890.84,2666.63,2700.61,3581.38,3376.96,2436.14,2623.82,2112.94,1827.71,1843.63,1667.29,2357.83
Coal,256.86,231.34,218.56,138.00,196.04,160.92,242.42,405.37,675.86,782.42,692.22,765.73,397.84
Other,537.73,542.59,423.14,435.06,385.65,330.22,409.62,395.89,345.14,358.46,383.36,323.74,405.49
Petroleum,38.82,41.57,58.03,45.60,72.92,62.93,77.56,81.81,76.94,54.20,58.49,41.53,59.28


### Heatmatrix

In [52]:
(r
 .div(1000)
 .sort_values(by = 12, ascending = False)
 .style
    .format(precision=2)
    .background_gradient(cmap="RdYlGn", axis=1)
)

month,1,2,3,4,5,6,7,8,9,10,11,12,AVG
technology,,,,,,,,,,,,,
Natural Gas,14.97,11.00,8.05,7.37,6.95,9.81,16.67,15.03,13.99,13.77,11.06,12.03,11.74
Solar,3.09,3.86,4.64,6.14,7.32,7.88,7.36,7.43,6.85,5.57,4.10,3.59,5.66
AVG,3.14,2.92,2.80,2.79,3.09,3.50,4.23,4.06,3.71,3.35,2.80,2.83,3.27
Nuclear,2.26,2.26,2.27,1.36,1.40,2.27,2.27,2.23,2.11,2.19,2.27,2.26,2.09
Hydro,2.39,3.56,4.09,4.15,4.77,4.11,4.35,4.28,3.50,2.26,1.87,2.00,3.44
Wind,1.56,1.89,2.67,2.70,3.58,3.38,2.44,2.62,2.11,1.83,1.84,1.67,2.36
Coal,0.26,0.23,0.22,0.14,0.20,0.16,0.24,0.41,0.68,0.78,0.69,0.77,0.40
Other,0.54,0.54,0.42,0.44,0.39,0.33,0.41,0.40,0.35,0.36,0.38,0.32,0.41
Petroleum,0.04,0.04,0.06,0.05,0.07,0.06,0.08,0.08,0.08,0.05,0.06,0.04,0.06


## Pivot table for charts

In [54]:
import plotly.express as px

px.box?

Signature:
px.box(
    data_frame=None,
    x=None,
    y=None,
    color=None,
    facet_row=None,
    facet_col=None,
    facet_col_wrap=0,
    facet_row_spacing=None,
    facet_col_spacing=None,
    hover_name=None,
    hover_data=None,
    custom_data=None,
    animation_frame=None,
    animation_group=None,
    category_orders=None,
    labels=None,
    color_discrete_sequence=None,
    color_discrete_map=None,
    orientation=None,
    boxmode=None,
    log_x=False,
    log_y=False,
    range_x=None,
    range_y=None,
    points=None,
    notched=False,
    title=None,
    subtitle=None,
    template=None,
    width=None,
    height=None,
) -> plotly.graph_objs._figure.Figure
Docstring:
    In a box plot, rows of `data_frame` are grouped together into a
    box-and-whisker mark to visualize their distribution.

    Each box spans from quartile 1 (Q1) to quartile 3 (Q3). The second
    quartile (Q2) is marked by a line inside the box. By default, the
    whiskers correspond to the

In [58]:
#need to convert to long format for plotly
r_melt= r.melt(ignore_index = False).reset_index() # will lose columns if don't ignore index

In [67]:
idx = r.sort_values(by = "AVG", ascending = False).index

In [68]:
px.box(
    data_frame= r_melt,
    x= "value",
    y = "technology",
    color= "technology",
    category_orders={"technology": idx},
)

In [69]:
px.area(
   data_frame= r_melt,
   x= "month",
   y = "value",
   color= "technology",
)

In [70]:
px.bar(
   data_frame= r_melt,
   x= "month",
   y = "value",
   color= "technology",
)